# 01 - Data scraping

* This notebook contains the code to download the official PDF files (one for each parliamentary session) directly from the Congress's website.

* It also fetches the date of each parliamentary session, which we store in dates.csv (*data/external*).

* Every file is named according to the parliamentary session numbering, where the first number indicates the legislative term, and the second one indicates the session number.
    > Ex: The file *PL_1_2* contains the second session from the first term.

* Due to the large number of files, only a few samples are found in the repo (*data/samples/pdfs*).

> **Inputs:** none (legislative terms range can be adjusted).

> **Outputs:** PDF files to `../data/samples/pdfs/` and `../data/external/dates.csv` (maps *sessions* to *dates*).




In [ ]:
from pathlib import Path
BASE_URL = "https://www.congreso.es/es/busqueda-de-publicaciones"

# Change the destination path to store the PDF files HERE:
DEST_PDF_DIR = Path('..') / 'data' / 'samples' / 'pdfs' 

DEST_PDF_DIR.mkdir(parents=True, exist_ok=True)

# Change the destination path to store the dates HERE:
DEST_DATES_CSV = Path('..') / 'data' / 'external' / 'dates.csv'

# Choose the legislative term range HERE:
LEGIS_RANGE = range(1, 16)  # ejemplo: terms 1 to 15

MAX_PAGES_PER_LEG = 200 # 
SLEEP_BETWEEN_REQUESTS = 0.2 # Seconds waiting in between each request

HEADERS = {
    'Accept': 'application/json, text/javascript, */*; q=0.01',
    'Accept-Encoding': 'gzip, deflate, br, zstd',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8',
    'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'Origin': 'https://www.congreso.es',
    'Referer': 'https://www.congreso.es/es/busqueda-de-publicaciones',
    'Sec-Fetch-Dest': 'empty',
    'Sec-Fetch-Mode': 'cors',
    'Sec-Fetch-Site': 'same-origin',
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'X-Requested-With': 'XMLHttpRequest'
}
TIMEOUT = 10 # Seconds waiting for a web response before throwing an error

In [ ]:
# Imports
import requests
import time
import logging
import pandas as pd
logging.getLogger().setLevel(logging.INFO)

In [ ]:
# When we make a POST request to this URL, we need to give this payload to access the sessions we want
# We also have include the term we want to target, and there's many pages for each one, so we have to iterate over them

PAYLOAD_TEMPLATE = {
    'p_p_id':'publicaciones',
    'p_p_lifecycle': 2,
    'p_p_state': 'normal',
    'p_p_mode': 'view',
    'p_p_resource_id': 'filtrarListado',
    'p_p_cacheability': 'cacheLevelPage',
    '_publicaciones_serie': 'Pleno',
    '_publicaciones_seccion': 'Congreso',
    '_publicaciones_tipoBusqueda': '0',
    '_publicaciones_publicacion': 'D'
}

In [ ]:
# Checking the connection to the URL
try:
    logging.info(' Testing GET request')
    r_test = requests.get(BASE_URL, headers=HEADERS, timeout=TIMEOUT)
    logging.info(f' GET status code: {r_test.status_code}')

    logging.info(' Testing POST request with minimal params...')
    test_payload = PAYLOAD_TEMPLATE.copy()
    test_payload['_publicaciones_legislatura'] = 1
    test_payload['_publicaciones_paginaActual'] = 1
    r_post = requests.post(BASE_URL, data=test_payload, headers=HEADERS, timeout=TIMEOUT)
    logging.info(f' POST status code: {r_post.status_code}')

    if r_post.status_code == 200:
        data = r_post.json()
        logging.info(f' ✓ Succesfull fetch')
    else:
        logging.warning(f'Status code {r_post.status_code}. Puede ser un problema con el sitio.')
except requests.exceptions.Timeout:
    logging.error(f' ✗ Timeout: URL exceeded timeout')
except requests.exceptions.ConnectionError as e:
    logging.error(f' ✗ Connection error: perhaps URL has changed. {e}')
except Exception as e:
    logging.error(f' ✗ Unexpected error: {e}')

In [ ]:
# We get a JSON file for a specific term and page (there will be many pages for each term and many sessions within each JSON)
def fetch_page_json(legislatura, pagina):
    payload = PAYLOAD_TEMPLATE.copy()
    payload['_publicaciones_legislatura'] = legislatura 
    payload['_publicaciones_paginaActual'] = pagina
    try:
        r = requests.post(BASE_URL, data=payload, headers=HEADERS, timeout=TIMEOUT)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        logging.warning(f'fetch_page_json failed for leg={legislatura} page={pagina}: {e}')
        return None

In [ ]:
# Given a session's url, we download the PDF file to a given path
def download_pdf_file(url, dest_path):
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        r.raise_for_status()
        with open(dest_path, 'wb') as f:
            f.write(r.content)
        return True
    except Exception as e:
        logging.warning(f'download failed {url}: {e}')
        return False

In [ ]:
# Main loop: go through each term and page, fetch the JSON, extract the PDF url and metadata, 
# download the PDF files and save the date in a dictionary to later store in a CSV file. 

dates = {}
for leg in LEGIS_RANGE:
    logging.info(f' Processing term {leg}')
    for page in range(1, MAX_PAGES_PER_LEG + 1):
        js = fetch_page_json(leg, page)
        if not js:
            break
        found_document = False
        for k, v in js.items(): # We iterate over each session in the page
            if k.startswith('documento'):
                found_document = True
                ndia = v.get('ndia')
                fecha = v.get('fecha')
                diario = v.get('diario')
                if diario:
                    pdf_url = 'https://www.congreso.es' + diario if diario.startswith('/') else diario

                    # We format each filename with the aformentioned format
                    fname = f'PL_{leg}_{ndia}.pdf' if ndia else f'PL_{leg}_{page}.pdf' 
                    dest = DEST_PDF_DIR / fname

                    if not dest.exists():
                        ok = download_pdf_file(pdf_url, dest)
                        time.sleep(SLEEP_BETWEEN_REQUESTS)
                        
                if ndia and fecha:
                    dates[(leg, ndia)] = fecha
        if not found_document:
            break

# Guardar fechas en CSV
rows = [(leg, ndia, fecha) for (leg, ndia), fecha in dates.items()]
df = pd.DataFrame(rows, columns=['leg', 'ndia', 'fecha'])

df.to_csv(DEST_DATES_CSV, index=False, header=False)